# Multi-Agent Research System

[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/code/genieincodebottle/multi-agent-research-system)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/genieincodebottle/aiml-companion/blob/main/projects/ai-agents-project/notebooks/Multi_Agent_Research.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-View_Source-blue?logo=github)](https://github.com/genieincodebottle/aiml-companion/tree/main/projects/ai-agents-project)

**Author:** [genieincodebottle](https://github.com/genieincodebottle) | **Last Updated:** March 2026

> LangGraph pipeline: Researcher -> Analyst -> Writer -> Fact-Checker with guardrails.

<div class="alert alert-info">
<strong>What you will learn:</strong>
<ul>
<li>Build a 4-agent research pipeline using <strong>LangGraph StateGraph</strong></li>
<li>Implement <strong>guardrails</strong>: PII detection, token budget enforcement, URL validation</li>
<li>Use <strong>Tavily</strong> for grounded web search (no hallucination)</li>
<li>Add <strong>fact-checking</strong> with citation tracking</li>
</ul>
</div>

<div class="alert alert-warning">
<strong>Requires API keys:</strong>
<ul>
<li><strong>Google API key</strong> - <a href="https://aistudio.google.com/app/apikey">Get one here</a> (free tier available for Gemini 2.5 Flash Lite)</li>
<li><strong>Tavily API key</strong> - <a href="https://app.tavily.com/">Get one here</a> (free tier: 1,000 searches/month)</li>
</ul>
</div>

**Run cells in order from top to bottom.** Each cell depends on the previous ones.

---

<a id="toc"></a>
## Table of Contents

1. [Setup](#setup) - Install LangGraph, LangChain, Tavily
2. [Set API Keys](#api-keys) - Configure Google and Tavily credentials
3. [Guardrails](#guardrails) - PII detection, budget tracking, URL validation
4. [Agent Nodes + Pipeline](#pipeline) - Build the 4-agent StateGraph
5. [Run the Pipeline](#run) - Execute a research query
6. [Key Takeaways](#takeaways) - Architecture patterns and lessons

---

<a id="setup"></a>
## 1. Setup

In [ ]:
!pip install langgraph langchain-google-genai langchain-community tavily-python requests -q

---

<a id="api-keys"></a>
## 2. Set API Keys

In [ ]:
import os
from getpass import getpass
os.environ["GOOGLE_API_KEY"] = getpass("Google API key: ")
os.environ["TAVILY_API_KEY"] = getpass("Tavily API key: ")

# Validate keys
assert len(os.environ["GOOGLE_API_KEY"]) > 10, "Google API key looks too short"
assert len(os.environ["TAVILY_API_KEY"]) > 10, "Tavily key looks too short"
print("Keys set and validated!")

---

<a id="guardrails"></a>
## 3. Guardrails

<div class="alert alert-warning">
<strong>Why guardrails?</strong> LLM-powered agents can leak PII, exceed token budgets, and cite broken URLs. These three guardrails run at every step of the pipeline:
<ul>
<li><strong>PII detection</strong> - Regex patterns for email, phone, SSN</li>
<li><strong>Token budget</strong> - Hard limit (50K tokens) prevents runaway costs</li>
<li><strong>URL validation</strong> - HEAD request to verify sources are live</li>
</ul>
</div>

In [ ]:
import re, requests, logging
logger = logging.getLogger(__name__)

PII_PATTERNS = {
    "email": re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"),
    "phone": re.compile(r"\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b"),
    "ssn": re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
}
TOKEN_BUDGET = 50000

def detect_pii(text):
    return {t: p.findall(text) for t, p in PII_PATTERNS.items() if p.findall(text)}

def scrub_pii(text):
    found = []
    for t, p in PII_PATTERNS.items():
        if p.search(text):
            found.append(t)
            text = p.sub(f"[REDACTED_{t.upper()}]", text)
    return text, found

def check_budget(tokens, budget=TOKEN_BUDGET):
    return tokens < budget

def validate_url(url, timeout=5):
    try:
        return requests.head(url, timeout=timeout, allow_redirects=True).status_code < 400
    except:
        return False

# Test
print(f"PII: {detect_pii('john@test.com 555-123-4567')}")
print(f"Budget (1K): {check_budget(1000)}, Budget (60K): {check_budget(60000)}")

<div class="alert alert-success">
<strong>Guardrails verified:</strong>
<ul>
<li>PII detection correctly identifies email addresses and phone numbers</li>
<li>Budget check correctly flags when token count exceeds the 50K limit</li>
<li>These functions will be called inside each agent node to enforce safety at every step</li>
</ul>
</div>

---

<a id="pipeline"></a>
## 4. Agent Nodes + Pipeline

<div class="alert alert-info">
<strong>Pipeline architecture:</strong>
<ol>
<li><strong>Researcher</strong> - Uses Tavily search to find relevant web sources</li>
<li><strong>Analyst</strong> - Extracts key claims with source attribution and confidence levels</li>
<li><strong>Writer</strong> - Produces a structured research report using ONLY the extracted claims</li>
<li><strong>Fact-Checker</strong> - Verifies claims against sources, flags unsupported ones, scrubs PII</li>
</ol>
All agents share a <code>ResearchState</code> TypedDict that flows through the graph.
</div>

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults

class ResearchState(TypedDict):
    query: str
    sources: list[dict]
    key_claims: list[dict]
    draft: str
    fact_check: list[dict]
    final_report: str
    token_count: int
    errors: list[str]

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)
search = TavilySearchResults(max_results=5)

def researcher(state):
    if not check_budget(state.get("token_count", 0)):
        return {"errors": state.get("errors", []) + ["Budget exceeded"]}
    results = search.invoke(state["query"])
    sources = [{"title": r.get("title",""), "url": r["url"], "snippet": r["content"][:500]} for r in results]
    return {"sources": sources, "token_count": state.get("token_count", 0) + 500}

def analyst(state):
    if not state.get("sources"):
        print("WARNING: No sources found. The writer will produce an empty report.")
        return {"key_claims": [], "errors": state.get("errors", []) + ["No sources returned by researcher"]}
    text = "\n".join(f"[{i+1}] {s['title']}: {s['snippet']}" for i, s in enumerate(state["sources"]))
    resp = llm.invoke(f"Extract 5-8 key claims. Format: claim | source | confidence\n\n{text}")
    claims = [{"claim": p[0].strip(), "source_idx": p[1].strip(), "confidence": p[2].strip() if len(p)>2 else "medium"}
              for line in resp.content.split("\n") for p in [line.split("|")] if len(p) >= 2]
    tokens = resp.usage_metadata.get("total_tokens", 1000) if hasattr(resp, 'usage_metadata') and resp.usage_metadata else 1000
    return {"key_claims": claims, "token_count": state.get("token_count", 0) + tokens}

def writer(state):
    claims = "\n".join(f"- {c['claim']} [Source {c['source_idx']}]" for c in state["key_claims"])
    sources = "\n".join(f"[{i+1}] {s['title']} - {s['url']}" for i, s in enumerate(state["sources"]))
    resp = llm.invoke(f"Write a research report using ONLY these claims.\n\nClaims:\n{claims}\n\nSources:\n{sources}")
    tokens = resp.usage_metadata.get("total_tokens", 1500) if hasattr(resp, 'usage_metadata') and resp.usage_metadata else 1500
    return {"draft": resp.content, "token_count": state.get("token_count", 0) + tokens}

def fact_checker(state):
    summaries = "\n".join(f"[{i+1}] {s['snippet'][:200]}" for i, s in enumerate(state["sources"]))
    check = llm.invoke(f"Verify claims. For unsupported: UNSUPPORTED: <claim>\n\nReport:\n{state['draft'][:2000]}\n\nSources:\n{summaries}")
    revised = state["draft"]
    for line in check.content.split("\n"):
        if line.strip().startswith("UNSUPPORTED:"):
            claim = line.replace("UNSUPPORTED:", "").strip()
            if claim and claim in revised:
                revised = revised.replace(claim, "[NEEDS CITATION] " + claim)
    revised, pii = scrub_pii(revised)
    tokens = check.usage_metadata.get("total_tokens", 1000) if hasattr(check, 'usage_metadata') and check.usage_metadata else 1000
    return {"final_report": revised, "token_count": state.get("token_count", 0) + tokens}

# Build graph
graph = StateGraph(ResearchState)
for name, fn in [("researcher", researcher), ("analyst", analyst), ("writer", writer), ("fact_checker", fact_checker)]:
    graph.add_node(name, fn)
graph.set_entry_point("researcher")
graph.add_edge("researcher", "analyst")
graph.add_edge("analyst", "writer")
graph.add_edge("writer", "fact_checker")
graph.add_edge("fact_checker", END)
app = graph.compile()
print("Pipeline built: researcher -> analyst -> writer -> fact_checker")

<div class="alert alert-info">
<strong>Pipeline built.</strong> The StateGraph defines a linear flow: researcher -> analyst -> writer -> fact_checker -> END. Each node reads from and writes to the shared <code>ResearchState</code>. LangGraph handles the execution order and state passing automatically.
</div>

---

<a id="run"></a>
## 5. Run the Pipeline

Execute the full research pipeline on a sample query. The pipeline will search the web, extract claims, write a report, and fact-check it.

In [ ]:
result = app.invoke({
    "query": "What are the latest trends in AI agents for 2025?",
    "sources": [], "key_claims": [], "draft": "",
    "fact_check": [], "final_report": "",
    "token_count": 0, "errors": []
})

print(f"Sources: {len(result['sources'])}")
print(f"Claims: {len(result['key_claims'])}")
print(f"Tokens: {result['token_count']} / {TOKEN_BUDGET}")
print(f"\n{'='*60}\nFINAL REPORT:\n{'='*60}")
print(result['final_report'][:2000])

<div class="alert alert-success">
<strong>Pipeline Results:</strong>
<ul>
<li><strong>Sources</strong> - Tavily returns 5 relevant web pages with titles, URLs, and content snippets</li>
<li><strong>Claims</strong> - The analyst extracts 5-8 key claims with source attribution and confidence levels</li>
<li><strong>Token usage</strong> - Total tokens consumed across all 4 agents (should be well under 50K budget)</li>
<li><strong>Final report</strong> - Fact-checked, PII-scrubbed research report with citations</li>
<li>Any [NEEDS CITATION] tags indicate claims the fact-checker could not verify against the provided sources</li>
</ul>
</div>

---

<a id="takeaways"></a>
## 6. Key Takeaways

<div class="alert alert-success">
<strong>Architecture patterns:</strong>
<ul>
<li><strong>LangGraph StateGraph</strong> - Shared TypedDict state flows between agents, enabling clean data handoff</li>
<li><strong>Guardrails at every step</strong> - PII scrubbing + budget enforcement prevents costly mistakes</li>
<li><strong>Fact-checker</strong> - Flags unsupported claims with [NEEDS CITATION] for human review</li>
<li><strong>Error accumulation</strong> - Errors are appended to state (not thrown), keeping the pipeline running</li>
<li><strong>Token budget</strong> - 50K hard limit prevents runaway API costs</li>
</ul>
</div>

<div class="alert alert-info">
<strong>Next steps:</strong>
<ul>
<li>Add a <strong>Human-in-the-Loop</strong> node for editorial review before publishing</li>
<li>Implement <strong>parallel research</strong> - multiple Researcher nodes for different subtopics</li>
<li>Add <strong>memory</strong> - store past research for follow-up queries</li>
<li>Try <strong>Claude</strong> or other LLM providers as the backbone</li>
</ul>
</div>

---

<div class="alert alert-info">
<strong>Found this notebook useful?</strong> Give it an upvote on Kaggle! Have questions or suggestions? Leave a comment below or open an issue on <a href="https://github.com/genieincodebottle/aiml-companion">GitHub</a>.
</div>